# Mini-World Model: Predict the Future! ⚽🌎
**AI Builders Lab | TensorFlow | Google Colab**

**Question:** Can a neural network learn to predict where a moving ball will go after we apply a force?

This notebook builds a tiny **action-conditioned dynamics world model**. You will generate physics data, train a neural network, compare real and predicted trajectories, measure errors, and imagine two different futures.

**Instructions:** In Colab, choose **Runtime → Run all**. CPU is enough. No external dataset or installation required.

**Learning goals:** state, action, next-state prediction, supervised learning, rollout, prediction error, counterfactual actions.

> This is a simplified state-based dynamics model, not an image/video generator, and it does not yet decide which actions to take.

## 1. What is the world model learning?
A ball has a state with 4 numbers:

$$s_t=[x_t,y_t,v_{x,t},v_{y,t}]$$

We choose an action with 2 numbers (the acceleration command):

$$a_t=[a_{x,t},a_{y,t}]$$

We want the AI to learn:

$$\hat{s}_{t+1}=f_\theta(s_t,a_t)$$

The neural network sees **6 inputs** (state + action) and predicts **4 outputs** (change in state). Its predicted next state is the current state plus that change.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# These are the easiest settings to experiment with!
DT = 0.1
DRAG = 0.12
TRAIN_SAMPLES = 12000
EPOCHS = 35
BATCH_SIZE = 256
ROLLOUT_STEPS = 70

print("TensorFlow:", tf.__version__)

## 2. Create our tiny physics world
The **real world** is a simple 2D simulator, with velocity, damping, and acceleration.

$$v_{t+1}=(1-\lambda\Delta t)v_t+a_t\Delta t$$

$$p_{t+1}=p_t+v_{t+1}\Delta t$$

The neural network is **never shown these equations**. It only sees example transitions generated by them.

There are no walls or collisions in this introductory experiment.

In [ ]:
def real_world_step(state, action):
    """Ground-truth physics for one sample or a batch of samples."""
    position = state[..., :2]
    velocity = state[..., 2:]
    next_velocity = (1.0 - DRAG * DT) * velocity + action * DT
    next_position = position + next_velocity * DT
    return np.concatenate([next_position, next_velocity], axis=-1).astype("float32")

rng = np.random.default_rng(SEED)

states = np.concatenate([
    rng.uniform(-3, 3, size=(TRAIN_SAMPLES, 2)),
    rng.uniform(-2, 2, size=(TRAIN_SAMPLES, 2)),
], axis=1).astype("float32")

actions = rng.uniform(-2, 2, size=(TRAIN_SAMPLES, 2)).astype("float32")
next_states = real_world_step(states, actions)

X = np.concatenate([states, actions], axis=1)  # 6 inputs
Y = next_states - states                       # 4 target changes

indices = rng.permutation(TRAIN_SAMPLES)
cut = int(TRAIN_SAMPLES * 0.8)
training_ids, validation_ids = indices[:cut], indices[cut:]
X_train, Y_train = X[training_ids], Y[training_ids]
X_val, Y_val = X[validation_ids], Y[validation_ids]

print("Input shape:", X.shape)
print("Target shape:", Y.shape)
print("Example state:", states[0].round(3))
print("Example action:", actions[0].round(3))
print("Next true state:", next_states[0].round(3))

## 3. Train the AI to predict state changes
We start with a tiny **multilayer perceptron (MLP)**, so the concept is easy to understand.

**Architecture:** 6 inputs → 64 neurons → 64 neurons → 4 outputs.

It learns by minimizing the mean squared error (MSE) between predicted changes and true changes.

**Question:** Why predict a change in state instead of directly predicting the full next state?

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(6,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(4),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    loss="mse",
)
history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=0,
)

print("Final train MSE:", history.history["loss"][-1])
print("Final validation MSE:", history.history["val_loss"][-1])

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error (log scale)")
plt.title("Training a world model")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. One-step prediction
We give the trained network a new state and action and compare its prediction with our simulator.

In [ ]:
current = np.array([0.5, -0.5, 1.0, 0.4], dtype="float32")
action = np.array([0.3, -1.2], dtype="float32")

feature = np.concatenate([current, action])[None, :]
predicted_change = model(feature, training=False).numpy()[0]
predicted_next = current + predicted_change
actual_next = real_world_step(current, action)

print("Current state:    ", current.round(4))
print("Action:           ", action.round(4))
print("Real next state:  ", actual_next.round(4))
print("AI next state:    ", predicted_next.round(4))
print("Absolute errors:  ", np.abs(predicted_next - actual_next).round(5))

## 5. Imagine the future (multi-step rollout)
Now the model predicts many steps. It uses its **own last prediction** as the next input, instead of receiving the real state each time.

This lets the model imagine a future trajectory—but errors can accumulate over time.

In [ ]:
def action_schedule(t):
    """Both the real world and AI receive the same action at time t."""
    return np.array([
        np.cos(0.12 * t),
        0.8 * np.sin(0.15 * t),
    ], dtype="float32")

def rollout(initial_state, steps, policy):
    real = [initial_state.astype("float32").copy()]
    imagined = [initial_state.astype("float32").copy()]
    for t in range(steps):
        action = np.asarray(policy(t), dtype="float32")
        real.append(real_world_step(real[-1], action))
        features = np.concatenate([imagined[-1], action])[None, :]
        change = model(features, training=False).numpy()[0]
        imagined.append(imagined[-1] + change)
    return np.array(real), np.array(imagined)

start = np.array([-1.0, -0.8, 0.9, 1.1], dtype="float32")
real, imagined = rollout(start, ROLLOUT_STEPS, action_schedule)

plt.figure(figsize=(8, 6))
plt.plot(real[:, 0], real[:, 1], linewidth=3, label="Real physics")
plt.plot(imagined[:, 0], imagined[:, 1], "--", linewidth=2.5,
         label="World model prediction")
plt.scatter(real[0, 0], real[0, 1], s=90, label="Start")
plt.scatter(real[-1, 0], real[-1, 1], marker="x", s=100, label="Real end")
plt.axis("equal")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Real trajectory vs predicted future")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. How does the error grow?
At each time step, measure the Euclidean distance between the true ball position and the predicted ball position:

$$e_t=\sqrt{(x_t-\hat{x}_t)^2+(y_t-\hat{y}_t)^2}$$

**Question:** Why might a tiny error become larger after 50 prediction steps?

In [ ]:
position_error = np.linalg.norm(real[:, :2] - imagined[:, :2], axis=1)

print("Mean position error:", round(float(position_error.mean()), 5))
print("Final position error:", round(float(position_error[-1]), 5))

plt.figure(figsize=(8, 4))
plt.plot(np.arange(len(position_error)) * DT, position_error)
plt.xlabel("Future time (seconds)")
plt.ylabel("Position error")
plt.title("Error over time")
plt.grid(alpha=0.3)
plt.show()

## 7. Counterfactual futures: What if we push differently?
The exact same initial state can lead to different outcomes depending on the action.

We ask the learned model to imagine **pushing right** and **pushing up**, then compare both imagined futures against the true simulator.

This is called **action-conditioned prediction**.

In [ ]:
push_right = lambda t: np.array([1.3, 0.0], dtype="float32")
push_up = lambda t: np.array([0.0, 1.3], dtype="float32")
same_start = np.array([0.0, 0.0, 0.1, 0.2], dtype="float32")

real_right, ai_right = rollout(same_start, 55, push_right)
real_up, ai_up = rollout(same_start, 55, push_up)

plt.figure(figsize=(8, 6))
plt.plot(real_right[:, 0], real_right[:, 1], label="Real: push right")
plt.plot(ai_right[:, 0], ai_right[:, 1], "--", label="AI: push right")
plt.plot(real_up[:, 0], real_up[:, 1], label="Real: push up")
plt.plot(ai_up[:, 0], ai_up[:, 1], "--", label="AI: push up")
plt.scatter([0], [0], s=70, marker="o", label="Start")
plt.axis("equal")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Two actions, two imagined futures")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Bonus: Animate the real and imagined ball
This animation is optional. It uses Matplotlib and the standard Colab display, with no extra dependencies.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

fig, ax = plt.subplots(figsize=(7, 5))
ax.set_xlim(min(real[:, 0].min(), imagined[:, 0].min()) - 0.5,
            max(real[:, 0].max(), imagined[:, 0].max()) + 0.5)
ax.set_ylim(min(real[:, 1].min(), imagined[:, 1].min()) - 0.5,
            max(real[:, 1].max(), imagined[:, 1].max()) + 0.5)
ax.set_aspect("equal")
ax.grid(alpha=0.2)
dot_real, = ax.plot([], [], "o", markersize=12, label="Real")
dot_ai, = ax.plot([], [], "X", markersize=10, label="AI")
line_real, = ax.plot([], [], "-", alpha=0.5)
line_ai, = ax.plot([], [], "--", alpha=0.5)
ax.legend(loc="upper left")

def animate(frame):
    dot_real.set_data([real[frame, 0]], [real[frame, 1]])
    dot_ai.set_data([imagined[frame, 0]], [imagined[frame, 1]])
    line_real.set_data(real[:frame+1, 0], real[:frame+1, 1])
    line_ai.set_data(imagined[:frame+1, 0], imagined[:frame+1, 1])
    ax.set_title(f"Time: {frame * DT:.1f} s")
    return dot_real, dot_ai, line_real, line_ai

animation = FuncAnimation(fig, animate, frames=len(real), interval=90)
plt.close(fig)
display(HTML(animation.to_jshtml()))

## 9. Student experiments and discussion

1. **Less training data:** Set `TRAIN_SAMPLES = 300`. Does prediction accuracy decrease?
2. **Smaller network:** Change both `Dense(64)` layers to `Dense(8)`. What happens?
3. **Longer prediction:** Set `ROLLOUT_STEPS = 150`. Does the AI remain accurate?
4. **New forces:** Change the `action_schedule` function to apply different forces.
5. **Outside the training range:** Try forces much greater than 2 in magnitude. Why might the model struggle?
6. **From images to states:** How could a visual encoder convert camera images into a compact world state?
7. **World model vs VLA:** Does this notebook choose the best action for a goal? What is missing?

### Key takeaways

- A **world model** predicts how the world changes.
- It can be **action-conditioned**, imagining what would happen under different actions.
- A learned model can have prediction errors, especially during long rollouts or outside the training data.
- A **policy (such as a VLA)** chooses or generates actions; a **world model** predicts their consequences.
- A future **Experiment C** can add planning by trying different action sequences and choosing the best predicted outcome.

> **Exit ticket:** Explain in one sentence why predicting the future is different from choosing what to do.